# Minecraft World Generator

Generate a Minecraft animation from your 4DWCM `MinCell.lm` trajectory.

## Terminal (recommended)

From any Delta login node or Gateway terminal:

```bash
# Demo trajectory
bash /projects/bgvl/alfiaparvez/SummerSchool_2026/4DWCM/minecraft/run_generate_world.sh --shared

# Group run (after leader moved 4dwcm_run to Groups_4DWCM)
bash .../run_generate_world.sh --group Group1
```

Output: `/projects/bgvl/$USER/minecraft/Minecraft_Cell_Animation_World/`

## Jupyter (optional)

Shared env: `/projects/bgvl/SummerSchool_2026/conda-envs/minecraft`  
Kernel: **Python (bgvl Minecraft)** — not `lm_2.5_dev`

Copy `Minecraft_Cell_Animation_World` into your Minecraft `saves/` folder to play.

In [ ]:
GROUP = "Group1"               # Group1, Group2, or Group3 (or "shared" for Mar31_1 demo)
RUN_NAME = "4dwcm_7200"        # output folder under Data/ (ignored when GROUP="shared")

### Imports

In [ ]:
# Packages are pre-installed in the shared bgvl Minecraft conda env.
# Only run this if you are NOT using kernel "Python (bgvl Minecraft)":
# %pip install .

In [ ]:
import h5py
import numpy as np
from mcschematic_plus import MCSchematicPlus, read_tiff, read_mesh, read_npy
from mcschematic import Version

### Read the .lm file

In [ ]:
if GROUP == "shared":
    path = "/projects/bgvl/SummerSchool_2026/4DWCM/trajectory/Mar31_1/MinCell.lm"
else:
    path = (
        f"/projects/bgvl/Groups_4DWCM/{GROUP}/4dwcm_run/"
        f"Optimize_4DWCM_Minimal_Cell/Data/{RUN_NAME}/MinCell.lm"
    )

print(f"Reading: {path}")
traj = h5py.File(path, "r")

max_time = traj['Simulations']['0000001']['LatticeTimes'][-1]

print(max_time)

In [ ]:
# Code to get list of molecular species names
sNbin = traj['Parameters']['SpeciesNames']
sN = []
for n in sNbin:
    sN.append(n[0].decode("utf-8"))
print(f"sN: {sN}")

# Function to get particle index of a molecule with name “name” from the species names list
def getPartIdx(name):
    return int(sN.index(name)+1)

# Code to get Site type names for site lattice
siteNames = traj['Parameters'].attrs['siteTypeNames'].decode("utf-8").split(',')

def getSiteIdx(name):
    return int(siteNames.index(name))

siteNames

# Function to get particle coordinates from particle lattice at a given time in seconds
def getPartCoords(traj, spec, time=None):
    coords = []
    if time is None:
        t = int(traj['Simulations']['0000001']['LatticeTimes'][-1])
    else:
        t = time

    pL = np.array(traj['Simulations']['0000001']['Lattice']['000000{}'.format(str(t).zfill(4))])

    scoords = np.argwhere(pL==getPartIdx(spec))
    scoords = scoords.T[0:3].T
    coords.append(scoords)
    # print(coords)

    return coords

# Function to get coordinates for a given site type
def getSiteCoords(traj, site, time=None):
    coords = []

    if time is None:
        t = int(traj['Simulations']['0000001']['LatticeTimes'][-1])
    else:
        t = time
    # Needs 000000 0000
    pL = np.array(traj['Simulations']['0000001']['Sites'][f'000000{str(t).zfill(4)}'])
    scoords = np.argwhere(pL==getSiteIdx(site))
    scoords = scoords.T[0:3].T
    coords.append(scoords)

    # print(coords)
    return coords

### Generate Muliple Schem Files

In [ ]:
frame_num = 30
OUTPUT_PATH = "Minecraft_Cell_Animation_World/datapacks/cellgen/data/cellgen/structure/animate"

# Warning: Changing this will change your output, but it will result in structures being identified incorrectly in the animation.
colors = {
    # 0 : "air". This is excluded volume from the cell
    1 : "air", # cytoplasm
    2 : "air", # outer_cytoplasm. Can change to glass.
    3 : "red_stained_glass", # ribosomes
    4 : "red_concrete", # ribo_centers
    5 : "yellow_concrete", # DNA
    6 : "lime_stained_glass" # membrane
}

# ['extracellular', 'cytoplasm', 'outer_cytoplasm', 'ribosomes', 'ribo_centers', 'DNA', 'membrane']. We will drop the extracellular when generating schematics.
jump = int(max_time / frame_num)
print(f"jump: {jump}")
for k in range(frame_num + 1):
    voxels = np.array(traj['Simulations']['0000001']['Sites'][f"000000{str(k * jump).zfill(4)}"])

    mask_voxels = voxels[..., None] == np.arange(np.max(voxels) + 1)
    # Drop the first value (0) because it's air.
    mask_voxels = mask_voxels[..., 1:]

    schem = MCSchematicPlus()

    for i, component in enumerate(colors.keys()):
        vol = mask_voxels[..., i] # already bool
        # mask any previously placed components to avoid overlap
        for j in range(i):
            vol = vol & ~mask_voxels[..., j]
        schem.placeVolume(vol, colors[component])

    schem.saveNBT(f"{OUTPUT_PATH}/lm_sim{k}.nbt", Version.JE_1_20_1, shifted=False)

In [ ]:
print("Congrats! You may now copy \"Minecraft_Cell_Animation_World\" into your saves folder!")